In [ ]:
# IMPORTS
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)
os.chdir(Path.cwd().parent)

In [ ]:
# One-Step-Ahead PPC: Posterior predictive Moran's I (5 models)
# Full script (data loading, scaling, W construction included)
# Condition on observed y_{t-1}, simulate y_t
import numpy as np
import pandas as pd
import geopandas as gpd
import pyreadr
import pickle
from pathlib import Path
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from tqdm import tqdm
# Config
BASE_DIR = Path(r"path/to/snow/data-and-results")
DIST_TH = 0.22
period = 52

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

def logistic(x):
    return 1.0 / (1.0 + np.exp(-x))
# 1) Load snow data
snow = pyreadr.read_r(BASE_DIR/"snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords_all = snow.iloc[:, :2].to_numpy()
y_all = snow.iloc[:, 2:].to_numpy()
# 2) Build W (two largest components)
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_all[:,0], coords_all[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
W_full = (squareform(pdist(xy)) <= DIST_TH).astype(int)
np.fill_diagonal(W_full, 0)
W_full = csr_matrix(W_full)

n_comp, labels = connected_components(W_full, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

use_idx = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

W = W_full[use_idx][:, use_idx]
y = y_all[use_idx]

S, TT = y.shape
S0 = W.sum()

print("Using S =", S, "TT =", TT)
# Moran's I
def moran_I(x):
    x = x.astype(float)
    xc = x - x.mean()
    num = xc @ (W @ xc)
    den = xc @ xc
    if den <= 0:
        return np.nan
    return (S / S0) * (num / den)

I_true = np.array([moran_I(y[:,t]) for t in range(TT)])
# Time covariates
t_full = np.arange(1, TT+1)
t_scaled = (t_full - t_full.mean()) / t_full.std(ddof=0)

t_raw_steps = np.arange(1, TT)
t_trend_steps = t_scaled[:-1]
week_steps = (np.arange(TT-1) % period)

cos_steps = np.cos(2*np.pi*t_raw_steps/period)
sin_steps = np.sin(2*np.pi*t_raw_steps/period)

cov4 = np.column_stack([
    np.ones(TT-1),
    cos_steps,
    sin_steps,
    t_trend_steps
])

cov8 = np.column_stack([
    np.ones(TT-1), np.ones(TT-1),
    cos_steps, cos_steps,
    sin_steps, sin_steps,
    t_trend_steps, t_trend_steps
])
# Static covariates for factor models
lat = coords_all[use_idx][:,1]
lat = (lat - lat.mean()) / lat.std()

elev = pd.read_csv(BASE_DIR/"curr_elev.csv").iloc[:,3].to_numpy()[use_idx]
elev = (elev - elev.mean()) / elev.std()

snow_temp = pyreadr.read_r(BASE_DIR/"snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp = snow_temp.drop(index=no_nbs).iloc[:,2:].to_numpy()[use_idx]
temp = (temp - temp.mean()) / temp.std()
# Load posteriors
theta01_iid = np.load(BASE_DIR/"ind01.npz")["all_theta"]
theta10_iid = np.load(BASE_DIR/"ind10.npz")["all_theta"]

theta01_bym = np.load(BASE_DIR/"bym01.npz")["all_theta"]
theta10_bym = np.load(BASE_DIR/"bym10.npz")["all_theta"]

res01_fac = np.load(BASE_DIR/"bym01_cov.npz")
res10_fac = np.load(BASE_DIR/"bym10_cov.npz")
theta01_fac = res01_fac["all_theta"]
theta10_fac = res10_fac["all_theta"]

with open(BASE_DIR/"bym01_weekly.pkl","rb") as f:
    res01_week = pickle.load(f)
with open(BASE_DIR/"bym10_weekly.pkl","rb") as f:
    res10_week = pickle.load(f)

eta01_week = res01_week["all_eta"]
tau01_week = res01_week["all_tau"]
eta10_week = res10_week["all_eta"]
tau10_week = res10_week["all_tau"]

with open(BASE_DIR/"bym01_cov_weekly.pkl","rb") as f:
    res01_wf = pickle.load(f)
with open(BASE_DIR/"bym10_cov_weekly.pkl","rb") as f:
    res10_wf = pickle.load(f)

eta01_wf = res01_wf["all_eta"]
tau01_wf = res01_wf["all_tau"]
eta10_wf = res10_wf["all_eta"]
tau10_wf = res10_wf["all_tau"]

M = theta01_iid.shape[1]
N_REP = 100
draw_ids = np.random.choice(M, N_REP, replace=False)

I_iid  = np.zeros((TT, N_REP))
I_bym  = np.zeros((TT, N_REP))
I_fac  = np.zeros((TT, N_REP))
I_week = np.zeros((TT, N_REP))
I_wfac = np.zeros((TT, N_REP))

for j in tqdm(range(N_REP), desc="Full PPC"):

    m = draw_ids[j] 

    # ================= IID =================
    yrep = np.zeros((S, TT), dtype=int)
    yrep[:,0] = y[:,0]

    th01 = theta01_iid[:,m]
    th10 = theta10_iid[:,m]

    b01 = [th01[k*S:(k+1)*S] for k in range(4)]
    b10 = [th10[k*S:(k+1)*S] for k in range(4)]

    I_iid[0,j] = moran_I(yrep[:,0])

    for t in range(1,TT):

        phi01 = sum(cov4[t-1,k]*b01[k] for k in range(4))
        phi10 = sum(cov4[t-1,k]*b10[k] for k in range(4))

        p01 = logistic(phi01)
        p10 = logistic(phi10)

        prev = y[:,t-1]
        prob = np.where(prev==0, p01, 1-p10)

        yrep[:,t] = (np.random.rand(S) < prob)
        I_iid[t,j] = moran_I(yrep[:,t])

    # ================= BYM =================
    yrep = np.zeros((S, TT), dtype=int)
    yrep[:,0] = y[:,0]

    th01 = theta01_bym[:,m]
    th10 = theta10_bym[:,m]

    b01 = [th01[k*S:(k+1)*S] for k in range(8)]
    b10 = [th10[k*S:(k+1)*S] for k in range(8)]

    I_bym[0,j] = moran_I(yrep[:,0])

    for t in range(1,TT):

        phi01 = sum(cov8[t-1,k]*b01[k] for k in range(8))
        phi10 = sum(cov8[t-1,k]*b10[k] for k in range(8))

        p01 = logistic(phi01)
        p10 = logistic(phi10)

        prev = y[:,t-1]
        prob = np.where(prev==0, p01, 1-p10)

        yrep[:,t] = (np.random.rand(S) < prob)
        I_bym[t,j] = moran_I(yrep[:,t])

    # ================= BYM + Factor =================
    yrep = np.zeros((S, TT), dtype=int)
    yrep[:,0] = y[:,0]

    th01 = theta01_fac[:,m]
    th10 = theta10_fac[:,m]

    b01 = [th01[k*S:(k+1)*S] for k in range(8)]
    b10 = [th10[k*S:(k+1)*S] for k in range(8)]
    g01 = th01[8*S:8*S+3]
    g10 = th10[8*S:8*S+3]

    I_fac[0,j] = moran_I(yrep[:,0])

    for t in range(1,TT):

        base01 = sum(cov8[t-1,k]*b01[k] for k in range(8))
        base10 = sum(cov8[t-1,k]*b10[k] for k in range(8))

        tsc = t_scaled[t-1]
        fac = np.column_stack([
            tsc*lat,
            tsc*elev,
            tsc*temp[:,t-1]
        ])

        phi01 = base01 + fac @ g01
        phi10 = base10 + fac @ g10

        p01 = logistic(phi01)
        p10 = logistic(phi10)

        prev = y[:,t-1]
        prob = np.where(prev==0, p01, 1-p10)

        yrep[:,t] = (np.random.rand(S) < prob)
        I_fac[t,j] = moran_I(yrep[:,t])

    # ================= Weekly =================
    yrep = np.zeros((S, TT), dtype=int)
    yrep[:,0] = y[:,0]

    e01 = eta01_week[:,m]
    t01 = tau01_week[:,m]
    e10 = eta10_week[:,m]
    t10 = tau10_week[:,m]

    b01 = [e01[k*S:(k+1)*S] for k in range(8)]
    b10 = [e10[k*S:(k+1)*S] for k in range(8)]

    I_week[0,j] = moran_I(yrep[:,0])

    for t in range(1,TT):

        w = week_steps[t-1]
        phi01 = np.zeros(S)
        phi10 = np.zeros(S)

        for k in range(8):
            phi01 += cov8[t-1,k]*b01[k]*t01[k*52+w]
            phi10 += cov8[t-1,k]*b10[k]*t10[k*52+w]

        p01 = logistic(phi01)
        p10 = logistic(phi10)

        prev = y[:,t-1]
        prob = np.where(prev==0, p01, 1-p10)

        yrep[:,t] = (np.random.rand(S) < prob)
        I_week[t,j] = moran_I(yrep[:,t])

    # ================= Weekly + Factor =================
    yrep = np.zeros((S, TT), dtype=int)
    yrep[:,0] = y[:,0]

    e01 = eta01_wf[:,m]
    t01 = tau01_wf[:,m]
    e10 = eta10_wf[:,m]
    t10 = tau10_wf[:,m]

    b01 = [e01[k*S:(k+1)*S] for k in range(8)]
    b10 = [e10[k*S:(k+1)*S] for k in range(8)]
    g01 = e01[8*S:8*S+3]
    g10 = e10[8*S:8*S+3]

    I_wfac[0,j] = moran_I(yrep[:,0])

    for t in range(1,TT):

        w = week_steps[t-1]
        base01 = np.zeros(S)
        base10 = np.zeros(S)

        for k in range(8):
            base01 += cov8[t-1,k]*b01[k]*t01[k*52+w]
            base10 += cov8[t-1,k]*b10[k]*t10[k*52+w]

        tsc = t_scaled[t-1]
        fac = np.column_stack([
            tsc*lat,
            tsc*elev,
            tsc*temp[:,t-1]
        ])

        phi01 = base01 + fac @ g01
        phi10 = base10 + fac @ g10

        p01 = logistic(phi01)
        p10 = logistic(phi10)

        prev = y[:,t-1]
        prob = np.where(prev==0, p01, 1-p10)

        yrep[:,t] = (np.random.rand(S) < prob)
        I_wfac[t,j] = moran_I(yrep[:,t])

In [ ]:
# Plot posterior mean + 95% CI
# Split into chunks of 500 time points (6 figures total)
import numpy as np
import matplotlib.pyplot as plt

def mu(x):  return np.nanmean(x, axis=1)
def qlo(x): return np.nanquantile(x, 0.025, axis=1)
def qhi(x): return np.nanquantile(x, 0.975, axis=1)

TT = I_iid.shape[0]
chunk = 500
n_plots = 6

for i in range(n_plots):

    start = i * chunk
    end = min((i + 1) * chunk, TT)

    t_axis = np.arange(start, end)

    plt.figure(figsize=(14,6))

    # IID
    plt.fill_between(t_axis,
                     qlo(I_iid)[start:end],
                     qhi(I_iid)[start:end],
                     alpha=0.12)
    plt.plot(t_axis, mu(I_iid)[start:end], label="IID")

    # BYM
    plt.fill_between(t_axis,
                     qlo(I_bym)[start:end],
                     qhi(I_bym)[start:end],
                     alpha=0.12)
    plt.plot(t_axis, mu(I_bym)[start:end], label="BYM")

    # BYM+Factor
    plt.fill_between(t_axis,
                     qlo(I_fac)[start:end],
                     qhi(I_fac)[start:end],
                     alpha=0.12)
    plt.plot(t_axis, mu(I_fac)[start:end], label="BYM+Factor")

    # Weekly
    plt.fill_between(t_axis,
                     qlo(I_week)[start:end],
                     qhi(I_week)[start:end],
                     alpha=0.12)
    plt.plot(t_axis, mu(I_week)[start:end], label="Weekly")

    # Weekly+Factor
    plt.fill_between(t_axis,
                     qlo(I_wfac)[start:end],
                     qhi(I_wfac)[start:end],
                     alpha=0.12)
    plt.plot(t_axis, mu(I_wfac)[start:end], label="Weekly+Factor")

    # Observed
    plt.plot(t_axis, I_true[start:end],
             color="black", linewidth=2, label="Observed")

    plt.title(f"Moran's I (Time {start}–{end})")
    plt.xlabel("Time index")
    plt.ylabel("Moran's I")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Aggregated mean absolute difference from observed Moran's I
# Smaller = better
import numpy as np
import pandas as pd

def mu(x):
    return np.nanmean(x, axis=1)

# posterior means
mu_iid  = mu(I_iid)
mu_bym  = mu(I_bym)
mu_fac  = mu(I_fac)
mu_week = mu(I_week)
mu_wfac = mu(I_wfac)

# aggregated mean absolute difference
mad_iid  = np.nanmean(np.abs(mu_iid  - I_true))
mad_bym  = np.nanmean(np.abs(mu_bym  - I_true))
mad_fac  = np.nanmean(np.abs(mu_fac  - I_true))
mad_week = np.nanmean(np.abs(mu_week - I_true))
mad_wfac = np.nanmean(np.abs(mu_wfac - I_true))

results = pd.DataFrame({
    "Model": [
        "IID",
        "BYM",
        "BYM+Factor",
        "Weekly",
        "Weekly+Factor"
    ],
    "MeanAbsDiff": [
        mad_iid,
        mad_bym,
        mad_fac,
        mad_week,
        mad_wfac
    ]
}).sort_values("MeanAbsDiff")

print(results)

In [ ]:
# Save all Moran's I matrices into one pickle file
import pickle

save_dict = {
    "I_iid": I_iid,
    "I_bym": I_bym,
    "I_fac": I_fac,
    "I_week": I_week,
    "I_wfac": I_wfac,
    "I_true": I_true
}

with open(BASE_DIR/"Moran_I_5models.pkl", "wb") as f:
    pickle.dump(save_dict, f)

print("Saved Moran_I_5models.pkl")